# 03 — Instrumento de rotulagem cega

**O que faz:** cria (se não existir) `labels_poc.csv` a partir dos hashes em
`panels/`, em ordem embaralhada com seed fixa, e oferece uma interface `ipywidgets`
dentro do próprio notebook para o avaliador marcar comportamentos por painel.

**Consome:** `%run 00_core.ipynb`, `data/poc_threshold_validation/results/rules_in_poc.csv`
(para a lista de comportamentos), `data/poc_threshold_validation/panels/*.png` (gerados
por `02_panel_generator.ipynb`).

**Produz:** `data/poc_threshold_validation/labels_poc.csv` (instrumento preenchido,
salvo incrementalmente a cada clique -- nunca sobrescreve rótulos já feitos).

**Ordem de execução:** roda depois de `02_panel_generator.ipynb`. É para uso
**interativo** do avaliador -- esta execução automatizada só valida que a interface
abre sem erro e que `labels_poc.csv` é criado com as colunas corretas; **nenhuma
rotulagem real foi feita nesta sessão** (exige julgamento humano sobre os canais
brutos, que não deve ser simulado).

In [1]:
%run 00_core.ipynb
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime, timezone

PROJECT_ROOT : C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing
DATA_DIR     : C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation
Tracks       : ['charlotte_roval_2025', 'summit_point']
Drivers      : {'Rodrigo': 'Driver A', 'Tomaz': 'Driver B', 'Morsinaldo': 'Driver C', 'Thallys': 'Driver D', 'Igor': 'Driver E', 'Hilton': 'Driver F'}
5 comparisons: ['B vs. A', 'C vs. B', 'D vs. B', 'E vs. B', 'F vs. B']
Self-comparison label: 'B vs. B (self)'
Descriptor functions loaded. DESC_SPEC = {'PEDAL_ACTIVE_PCT': 5.0, 'MIN_BRAKE_RUN': 5}
align_lap_by_dist / assert_pedal_scale loaded.
12 rules, 17 tunable thresholds, 2 descriptor parameters.
  MRP_DT               =     -0.05
  BRAKE_EFF_RATIO      =      0.85
  TRAIL_RATIO          =       0.7
  LEGACY_DV            =        -2
  LEGACY_BRAKE_GATE    =         5
  BRAKE_DIST_TOL       =     0.005
  OVERBRAKE_DELTA      =        15
  ENTRY_SLOW_REL       =    -0.015
  ENTRY_BRAKE_GATE     =    

OK (60,130 samples, laps 0–12)
──────────────────────────────────────────────────────────────
  Lap Validity Report  (13 laps total)
──────────────────────────────────────────────────────────────
  ✅ Valid   : 9 laps
  🏆 Fastest : Lap 8  (01:19.767)
  ❌ Invalid : 4 laps
     Lap   0  16:39.700  → LapTime=999.7s | GPS=27.0%<100%
     Lap   1  01:22.067  → IQR_outlier
     Lap   3  01:22.783  → IQR_outlier
     Lap  12  01:22.633  → CompletedPct=0.938<0.995 | GPS=94.0%<100%
──────────────────────────────────────────────────────────────
[SCALE] pedal channels confirmed on a 0-100 scale (brake max = 78.5 %)
pair_cache      : 1 comparison-stints, 18 sector pairs.
pair_cache_self : 1 comparison-stints, 18 sector pairs.
Total activations (cross-driver, current scope): 47
Total activations (self, current scope)        : 31

Cache persisted under C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\cache
stint_slopes / pattern_stats / scenario_summary loade

In [2]:
rules_in_poc_path = RESULTS_DIR / "rules_in_poc.csv"
if not rules_in_poc_path.exists():
    raise FileNotFoundError(
        f"{rules_in_poc_path} not found -- run 01_gap_closure.ipynb first.")
rules_in_poc = pd.read_csv(rules_in_poc_path)
labeling_rows = rules_in_poc[rules_in_poc["in_labeling"]]
BEHAVIORS = sorted(labeling_rows["behavior"].unique().tolist())

missing_pt = [b for b in BEHAVIORS if b not in BEHAVIOR_LABEL_PT]
if missing_pt:
    raise KeyError(f"BEHAVIOR_LABEL_PT is missing a PT label for: {missing_pt}")

print(f"{len(BEHAVIORS)} behaviors in the labeling checklist (short PT labels, no rule name shown):")
for b in BEHAVIORS:
    print(f"  {b:32s} -> {BEHAVIOR_LABEL_PT[b]}")

10 behaviors in the labeling checklist (short PT labels, no rule name shown):
  brake_application_abrupt         -> Frenagem abrupta / instável
  brake_point_differs              -> Ponto de frenagem diferente
  entry_too_slow                   -> Entrada mais lenta que a referência
  excess_steering                  -> Esterçamento excessivo
  exit_slow_no_brake               -> Sai mais devagar sem frear mais
  no_trail_braking                 -> Sem trail braking / solta o freio cedo
  over_braking                     -> Frenagem mais forte que a referência
  rotation_after_vmin              -> Gira o carro depois do ponto mais lento
  throttle_hesitation_slow_exit    -> Hesita no acelerador na saída lenta
  throttle_with_steering           -> Acelera enquanto ainda esterça bastante


## Criação de `labels_poc.csv`

Hashes lidos de `panels/`, ordem embaralhada com seed fixa (`np.random.default_rng(20250813)`).
Se o arquivo já existir, carrega do disco e **só adiciona linhas para hashes novos**
(ex.: depois de reamostrar com o cache completo) -- rótulos já preenchidos nunca são
tocados.

In [3]:
LABELS_PATH = DATA_DIR / "labels_poc.csv"

panel_hashes = sorted(p.stem for p in PANELS_DIR.glob("*.png"))
if not panel_hashes:
    print(f"[WARN] no panels found under {PANELS_DIR} -- run 02_panel_generator.ipynb first.")

shuffle_rng = np.random.default_rng(20250813)
shuffled_hashes = list(panel_hashes)
shuffle_rng.shuffle(shuffled_hashes)

BASE_COLUMNS = ["hash", "labeled_at", "session_id"] + BEHAVIORS + ["unsure", "notes"]

if LABELS_PATH.exists():
    labels_df = pd.read_csv(LABELS_PATH)
    for col in BASE_COLUMNS:
        if col not in labels_df.columns:
            labels_df[col] = pd.NA
    existing = set(labels_df["hash"])
    new_hashes = [h for h in shuffled_hashes if h not in existing]
    if new_hashes:
        new_rows = pd.DataFrame({"hash": new_hashes})
        for col in BASE_COLUMNS:
            if col not in new_rows.columns:
                new_rows[col] = pd.NA
        labels_df = pd.concat([labels_df, new_rows[BASE_COLUMNS]], ignore_index=True)
        labels_df.to_csv(LABELS_PATH, index=False)
    print(f"Loaded existing {LABELS_PATH.name}: {len(labels_df)} rows "
          f"({len(new_hashes)} new hashes appended, existing labels untouched).")
else:
    labels_df = pd.DataFrame({"hash": shuffled_hashes})
    for col in BASE_COLUMNS:
        if col not in labels_df.columns:
            labels_df[col] = pd.NA
    labels_df = labels_df[BASE_COLUMNS]
    labels_df.to_csv(LABELS_PATH, index=False)
    print(f"Created {LABELS_PATH} with {len(labels_df)} rows, columns: {list(labels_df.columns)}")

display(labels_df.head())

Created C:\Users\to_fi\Documents\GitHub\Doutorado\Racing4all\Iracing\data\poc_threshold_validation\labels_poc.csv with 36 rows, columns: ['hash', 'labeled_at', 'session_id', 'brake_application_abrupt', 'brake_point_differs', 'entry_too_slow', 'excess_steering', 'exit_slow_no_brake', 'no_trail_braking', 'over_braking', 'rotation_after_vmin', 'throttle_hesitation_slow_exit', 'throttle_with_steering', 'unsure', 'notes']


,hash,labeled_at,session_id,brake_application_abrupt,brake_point_differs,entry_too_slow,excess_steering,exit_slow_no_brake,no_trail_braking,over_braking,rotation_after_vmin,throttle_hesitation_slow_exit,throttle_with_steering,unsure,notes
0,f4d89e34bc,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,75e6657d40,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2a66aee4a2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,8e5f6ffe34,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,bae2303733,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Instruções para o avaliador

- Rotule **apenas pelos canais brutos** exibidos no painel (speed, brake, throttle,
  steering, yaw rate). Não há tabela de descritores nem qualquer pista textual sobre
  qual regra motivou a amostragem daquele par.
- **Não abra `blind_map.csv` antes do fim da rotulagem.** Esse arquivo liga o hash à
  identidade (piloto, pista, stint, setor) e a qualquer ativação de regra -- abri-lo
  contamina o julgamento e invalida a validação de construto.
- Sessões de **no máximo 40 painéis** por vez, para evitar fadiga de julgamento.
- Preencha o campo `session_id` (ex.: `2026-09-14-sessao1`) antes de começar e mantenha
  o mesmo valor durante toda a sessão -- ele é salvo em cada rótulo.
- Marque `unsure` quando não tiver confiança na resposta, em vez de adivinhar.
- O progresso e os rótulos são salvos em disco a cada clique -- é seguro fechar e
  retomar depois de qualquer ponto.

In [4]:
session_id_box = widgets.Text(value="", placeholder="ex.: 2026-09-14-sessao1",
                              description="session_id:", style={"description_width": "initial"})

behavior_checks = {b: widgets.Checkbox(value=False, description=BEHAVIOR_LABEL_PT[b],
                                       indent=False, layout=widgets.Layout(width="420px"))
                   for b in BEHAVIORS}
unsure_check = widgets.Checkbox(value=False, description="Não tenho certeza", indent=False)
notes_box    = widgets.Textarea(value="", placeholder="notas (opcional)",
                                layout=widgets.Layout(width="420px", height="60px"))

progress_label = widgets.Label(value="")
image_box      = widgets.Image(format="png", layout=widgets.Layout(width="480px"))

btn_save_next = widgets.Button(description="Salvar + Próximo", button_style="success")
btn_skip      = widgets.Button(description="Pular", button_style="")
btn_back      = widgets.Button(description="Voltar", button_style="")

order = labels_df["hash"].tolist()
state = {"pos": 0}


def first_unlabeled_pos():
    for i, h in enumerate(order):
        row = labels_df.loc[labels_df["hash"] == h].iloc[0]
        if pd.isna(row["labeled_at"]) or row["labeled_at"] == "":
            return i
    return 0


def load_panel(pos):
    h = order[pos]
    png_path = PANELS_DIR / f"{h}.png"
    if png_path.exists():
        image_box.value = png_path.read_bytes()
    row = labels_df.loc[labels_df["hash"] == h].iloc[0]
    for b in BEHAVIORS:
        v = row.get(b, 0)
        behavior_checks[b].value = bool(v) if pd.notna(v) else False
    unsure_check.value = bool(row["unsure"]) if pd.notna(row.get("unsure")) else False
    notes_box.value = str(row["notes"]) if pd.notna(row.get("notes")) else ""
    progress_label.value = f"{pos + 1}/{len(order)}  (hash: {h})"


def save_current(pos):
    h = order[pos]
    idx = labels_df.index[labels_df["hash"] == h][0]
    labels_df.loc[idx, "labeled_at"] = datetime.now(timezone.utc).isoformat()
    labels_df.loc[idx, "session_id"] = session_id_box.value
    for b in BEHAVIORS:
        labels_df.loc[idx, b] = int(behavior_checks[b].value)
    labels_df.loc[idx, "unsure"] = int(unsure_check.value)
    labels_df.loc[idx, "notes"] = notes_box.value
    labels_df.to_csv(LABELS_PATH, index=False)


def on_save_next(_):
    save_current(state["pos"])
    state["pos"] = min(state["pos"] + 1, len(order) - 1)
    load_panel(state["pos"])


def on_skip(_):
    state["pos"] = min(state["pos"] + 1, len(order) - 1)
    load_panel(state["pos"])


def on_back(_):
    state["pos"] = max(state["pos"] - 1, 0)
    load_panel(state["pos"])


btn_save_next.on_click(on_save_next)
btn_skip.on_click(on_skip)
btn_back.on_click(on_back)

if order:
    state["pos"] = first_unlabeled_pos()
    load_panel(state["pos"])

ui = widgets.VBox([
    session_id_box,
    progress_label,
    image_box,
    widgets.VBox(list(behavior_checks.values())),
    unsure_check,
    notes_box,
    widgets.HBox([btn_back, btn_skip, btn_save_next]),
])
display(ui)
print("UI built. No labeling performed in this automated run -- "
      "interact with the widgets above to label for real.")

UI built. No labeling performed in this automated run -- interact with the widgets above to label for real.


## Status

Quantos painéis já foram rotulados, quantos marcados como `unsure`, e a distribuição
de cada comportamento entre os rótulos preenchidos até agora.

In [5]:
labeled_mask = labels_df["labeled_at"].notna() & (labels_df["labeled_at"] != "")
n_labeled = int(labeled_mask.sum())
n_unsure  = int(pd.to_numeric(labels_df.loc[labeled_mask, "unsure"], errors="coerce").fillna(0).sum())

print(f"Rotulados: {n_labeled}/{len(labels_df)}")
print(f"Unsure   : {n_unsure}/{max(n_labeled, 1)}")

if n_labeled > 0:
    dist = (pd.to_numeric(labels_df.loc[labeled_mask, BEHAVIORS].stack(), errors="coerce")
            .groupby(level=1).sum().reindex(BEHAVIORS).fillna(0).astype(int))
    print("\nDistribuição por comportamento (entre os rotulados):")
    display(dist.rename("count").reset_index().rename(columns={"index": "behavior"}))
else:
    print("\nNenhum rótulo ainda -- distribuição vazia (esperado: rotulagem real não "
          "foi executada nesta sessão).")

Rotulados: 0/36
Unsure   : 0/1

Nenhum rótulo ainda -- distribuição vazia (esperado: rotulagem real não foi executada nesta sessão).
